# RFP-02 | Mission simulation r04
Single active M400 + L3, identical standby system (not flown). LZ-01 highway parking.

Place the updated config beside this notebook. Dependencies: numpy, pandas, matplotlib, shapely, IPython; FFmpeg optional. Restart kernel and Run All. Outputs: `mission_outputs_r04/`. This pass selects a provisional operating point and implements payload coverage and schedule; formal comparative studies are deferred.

## 1. Engineering basis
Sources: [DJI L3 specifications](https://enterprise.dji.com/zenmuse-l3/specs), [L3 FAQ](https://enterprise.dji.com/zenmuse-l3/faq), [L3 product page](https://enterprise.dji.com/zenmuse-l3), [M400 specifications](https://enterprise.dji.com/matrice-400/specs), RFP-02 §§5, 7, 10 and Appendix A, project outline §4, concept slide 11. Manufacturer data accessed November 2026.

The 80–120 m surface-distance envelope is an assumption around a 100 m nominal flight plane, not verified terrain following; a DEM/DSM and legal AGL check are outstanding. 350 kHz and 16 returns are compatible; star-shaped scanning is selected for forest penetration (DJI product page). Published 120 m LiDAR accuracy (3 cm vertical, 4 cm horizontal) uses linear scanning and exposed checkpoints; it is not proof for star mode, vegetation or RGB accuracy.

For flat terrain, LiDAR width = 2H tan(40°). Dual cameras are assumed symmetrically tilted ±22.5° cross-track; each has 62° × 41.2° FOV, 12288 × 8192 pixels, JPEG 1 s minimum interval. Crop outer rays to 50°. Use the minimum surface distance for overlap and maximum for GSD. The config projects pinhole rays and computes a conservative outer-corner GSD using the largest singular value of the ground projection Jacobian. Side spacing uses 70% overlap of each individual camera width (not combined swath); forward overlap target 80%, speed capped at 10 m/s. These overlap targets are engineering assumptions.

M400: 9.740 kg with TB100, L3 + connector 1.745 kg, 15.8 kg MTOW. DJI's 53 min reference is H30T + AP100 at sea level, not L3. Assume 45 min to depletion and reserve 25% (33.75 min usable). Transit 12 m/s, climb/descent 2.5 min combined, 1 min additional IMU allowance per sortie (confirm actual Pilot 2 procedure), 6 s turn delay. Nine charged TB100 available for nominal sorties plus one spare, no on-site charging. Eight-hour day, 60 min survey control, 35 min setup/packdown, 25 min delay allowance and 7 min QA per sortie are assumptions. Standby aircraft not allocated flight time; payload availability and cost to be confirmed.

Water: preserve all terrestrial cells and a provisional 60 m shoreline water band for RGB; omit only deep interior of traced lake. Island retained. This is an interpretation of MR-01/02 and PR-04, not permission to omit other water or proof of shoreline completeness. Digitization needs validation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.path import Path as PlotPath
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display

import rfp02_config as cfg

output_dir = Path("mission_outputs_r04")
output_dir.mkdir(exist_ok=True)
export_video = False
show_plots = True

site_id = cfg.mission_site_id
from shapely.geometry import Polygon, LineString, box
from shapely.affinity import affine_transform
from shapely.ops import unary_union

# Explicit operating-point calculations; no fleet or launch-site comparison.
print(pd.Series({"Nominal altitude m":cfg.flight_height_above_base_m, "LiDAR spacing limit m":cfg.lidar_spacing_limit_m, "RGB spacing limit m":cfg.rgb_spacing_limit_m, "Chosen spacing m":cfg.line_spacing_m, "RGB speed limit m/s":cfg.rgb_speed_limit_mps, "Chosen speed m/s":cfg.survey_speed_mps, "Worst GSD cm":cfg.gsd_max_cm, "Mass kg":cfg.takeoff_mass_kg}).round(3))
assert cfg.gsd_max_cm <= cfg.rgb_gsd_target_cm
region = Polygon(cfg.corners)
lake = Polygon(cfg.lake_xy).buffer(0).difference(Polygon(cfg.island_xy).buffer(0))
deep = lake.buffer(-cfg.shoreline_water_buffer_m) if cfg.exclude_deep_water else Polygon()
target = region.difference(deep)
# Clip each strip against the target expanded across its assigned lane.
# This conservatively retains all assigned lane cells, including island/shore.
def clipped_route(bearing):
    a=np.deg2rad(bearing); u=np.array([np.sin(a),np.cos(a)]); q=np.array([u[1],-u[0]])
    local=affine_transform(target,[*u,*q,0,0])
    strips=[]
    for start,end in cfg.scan_grid(bearing,cfg.line_spacing_m):
        y=start@q; lo,hi=sorted([start@u,end@u]); half=cfg.line_spacing_m/2
        section=local.intersection(box(lo,y-half,hi,y+half))
        parts=list(section.geoms) if hasattr(section,"geoms") else [section]
        intervals=sorted([(g.bounds[0]-cfg.run_in_out_m,g.bounds[2]+cfg.run_in_out_m) for g in parts if not g.is_empty])
        merged=[]
        for l,r in intervals:
            if merged and l<=merged[-1][1]: merged[-1][1]=max(r,merged[-1][1])
            else: merged.append([l,r])
        if (end-start)@u<0: merged=[(r,l) for l,r in merged[::-1]]
        strips.extend(np.array([l*u+y*q,r*u+y*q]) for l,r in merged if abs(r-l)>1e-6)
    return np.asarray(strips)

site=cfg.sites[site_id]; fixed=cfg.access_minutes(site)+cfg.setup_packdown_min+cfg.survey_control_min_per_day+cfg.delay_allowance_min_per_day
plan=None
for bearing in cfg.bearings_deg:
    grid=clipped_route(bearing)
    for route_candidate in (grid,grid[::-1,::-1]):
        times,reach=cfg.sortie_times(route_candidate,site,cfg.airborne_endurance_min,cfg.c2_slant_limit_m)
        if times is None: continue
        times=times+cfg.imu_allowance_min_per_sortie
        times[times>cfg.usable_flight_min]=np.inf
        result=cfg.schedule(times,fixed)
        if result is not None and (plan is None or result[:2]<plan['result'][:2]):
            plan=dict(result=result,route=route_candidate,bearing=bearing,reach=reach)
if plan is None: raise ValueError("No feasible nominal schedule; review inputs")
base = cfg.sites[site_id].xy
route = plan["route"]
field_days, total_min, schedule = plan["result"]
print(f"{cfg.site_label(site_id)} | bearing {plan['bearing']} degrees | {len(route)} strips")
print(f"Flight ceiling per sortie: {cfg.usable_flight_min:.2f} min, including the landing-reserve deduction.")

2. Time-based trajectory

Each segment has a start and end time and a start and end position. The elapsed clock advances at the configured ground speeds on straight horizontal segments. A stationary turn delay is charged before each repositioning leg. The combined climb/descent budget is split equally and used to interpolate altitude. A survey segment starts triggering both RGB cameras immediately, then every configured photo interval until the segment ends. LiDAR records continuously during survey segments. The time origin is the start of field work. For multiple field days, the clock counts field time and omits overnight downtime.

In [ ]:
segments, photo_events, sortie_rows = [], [], []

def add_segment(phase, start, end, duration_s, z0=0, z1=0, strip=None):
    t0 = segments[-1]["end_s"] if segments else 0.0
    segments.append(dict(day=day, sortie=sortie, phase=phase, start_s=t0, end_s=t0 + duration_s,
                         east_start_m=start[0], north_start_m=start[1], height_start_m=z0,
                         east_end_m=end[0], north_end_m=end[1], height_end_m=z1,
                         distance_m=np.linalg.norm(end - start), strip=strip,
                         lidar_on=phase == "Survey", rgb_enabled=phase == "Survey"))
    if phase == "Survey":
        for offset in np.arange(0, duration_s, cfg.rgb_photo_interval_s):
            position = start + offset / duration_s * (end - start)
            photo_events.append((day, sortie, strip, t0 + offset, *position, z0, 2))

def fly(phase, start, end, speed, strip=None):
    height = cfg.flight_height_above_base_m
    add_segment(phase, start, end, np.linalg.norm(end - start) / speed, height, height, strip)

for day, day_plan in enumerate(schedule, 1):
    sortie = 0
    add_segment("Setup and checks", base, base, 30 * cfg.setup_packdown_min)
    add_segment("Survey control", base, base, 60 * cfg.survey_control_min_per_day)
    for sortie, (first, stop, optimizer_min) in enumerate(day_plan, 1):
        takeoff = segments[-1]["end_s"]
        first_event = len(segments)
        height = cfg.flight_height_above_base_m
        add_segment("Climb", base, base, cfg.climb_descent_min * 30, 0, height)
        add_segment("IMU allowance", base, base, 60*cfg.imu_allowance_min_per_sortie, height, height)
        fly("Outbound transit", base, route[first, 0], cfg.transit_speed_mps)
        for i in range(first, stop):
            start, end = route[i]
            if i > first:
                previous = route[i - 1, 1]
                add_segment("Turn", previous, previous, 60 * cfg.turn_delay_min, height, height)
                fly("Reposition", previous, start, cfg.transit_speed_mps)
            fly("Survey", start, end, cfg.survey_speed_mps, i + 1)
        fly("Return transit", route[stop - 1, 1], base, cfg.transit_speed_mps)
        add_segment("Descent", base, base, cfg.climb_descent_min * 30, height, 0)
        landing = segments[-1]["end_s"]
        actual_min = (landing - takeoff) / 60
        assert np.isclose(optimizer_min, actual_min)
        assert actual_min <= cfg.usable_flight_min + 1e-9
        legs = segments[first_event:]
        sortie_rows.append((day, sortie, first + 1, stop, takeoff / 60, landing / 60, actual_min,
                            cfg.usable_flight_min - actual_min, sum(s["distance_m"] for s in legs) / 1000))
        add_segment("Turnaround and data QA", base, base, 60 * cfg.turnaround_min)
    sortie = 0
    add_segment("Delay allowance", base, base, 60 * cfg.delay_allowance_min_per_day)
    add_segment("Packdown", base, base, 30 * cfg.setup_packdown_min)

timeline = pd.DataFrame(segments)
photos = pd.DataFrame(photo_events, columns=["day", "sortie", "strip", "time_s", "east_m", "north_m",
                                             "height_m", "images"])
sorties = pd.DataFrame(sortie_rows, columns=["Day", "Sortie", "First strip", "Last strip",
                                            "Takeoff min", "Landing min", "Airborne min", "Margin min",
                                            "Horizontal km"])
assert site_id == cfg.parking_id and cfg.access_minutes(cfg.sites[site_id]) == 0
assert np.isclose(timeline.end_s.iloc[-1] / 60, total_min)
assert np.allclose(timeline.end_s.iloc[:-1], timeline.start_s.iloc[1:])
for axis in ("east", "north", "height"):
    assert np.allclose(timeline[f"{axis}_end_m"].iloc[:-1], timeline[f"{axis}_start_m"].iloc[1:])
acquisition = timeline.loc[timeline.lidar_on]
assert acquisition.strip.tolist() == list(range(1, len(route) + 1))
display(sorties.round(2))
sorties.to_csv(output_dir / "sortie_schedule.csv", index=False)
timeline.to_csv(output_dir / "mission_timeline.csv", index=False)
photos.to_csv(output_dir / "rgb_triggers.csv", index=False)

## 3. Payload tasks and coverage
LiDAR is active continuously during survey legs; RGB paired exposures occur every second. Grid cells accumulate individual camera image footprints (conservative rectangles inside the projected trapezoids), strip counts and nominal emitted pulse density. Density assumes uniform distribution within the full 80° swath; no canopy or detection model. Ground returns and achieved accuracy remain unverified. Coverage time is the first nominal observation time, not the time at which a validated product becomes available.

In [ ]:
step=cfg.coverage_grid_m
x,y=np.meshgrid(np.arange(cfg.corners[:,0].min()+step/2,cfg.corners[:,0].max(),step),np.arange(cfg.corners[:,1].min()+step/2,cfg.corners[:,1].max(),step))
points=np.column_stack((x.ravel(),y.ravel())); points=points[PlotPath(cfg.corners).contains_points(points)]
water=PlotPath(cfg.lake_xy).contains_points(points)&~PlotPath(cfg.island_xy).contains_points(points)
deep_mask=np.zeros(len(points),bool)
for poly in (list(deep.geoms) if hasattr(deep,'geoms') else [deep]):
    if not poly.is_empty:
        mask=PlotPath(np.asarray(poly.exterior.coords)).contains_points(points)
        for hole in poly.interiors: mask &= ~PlotPath(np.asarray(hole.coords)).contains_points(points)
        deep_mask |= mask
rgb_required=~deep_mask | ~water
lidar_hits=np.zeros(len(points),int); rgb_hits=np.zeros(len(points),int); rgb_frames=np.zeros(len(points),int)
first_l=np.full(len(points),np.inf); first_r=first_l.copy()
for strip,(start,end) in enumerate(route,1):
    leg=acquisition.iloc[strip-1]; length=np.linalg.norm(end-start); u=(end-start)/length; q=np.array([u[1],-u[0]])
    along=(points-start)@u; cross=(points-start)@q
    hit=(along>=0)&(along<=length)&(abs(cross)<=cfg.lidar_width_m/2)
    lidar_hits+=hit; first_l[hit]=np.minimum(first_l[hit],leg.start_s+along[hit]/cfg.survey_speed_mps)
    # Count triggers analytically for each individual camera, using integer shutter indices.
    count=len(photos[photos.strip==strip]); dx=cfg.survey_speed_mps*cfg.rgb_photo_interval_s; half=cfg.rgb_min_along_m/2
    lower=np.maximum(0,np.ceil((along-half)/dx)).astype(int); upper=np.minimum(count-1,np.floor((along+half)/dx)).astype(int)
    visible=upper>=lower; strip_hit=np.zeros(len(points),bool)
    for sign in (-1,1):
        c=sign*cross; hit=visible&(c>=cfg.height_min_m*np.tan(cfg.theta_inner))&(c<=cfg.height_min_m*np.tan(cfg.theta_outer))
        rgb_frames[hit]+=upper[hit]-lower[hit]+1; strip_hit|=hit
        first_r[hit]=np.minimum(first_r[hit],leg.start_s+lower[hit]*cfg.rgb_photo_interval_s)
    rgb_hits+=strip_hit
nominal_pulses=cfg.lidar_pulse_rate_hz/(cfg.survey_speed_mps*2*cfg.height_max_m*np.tan(np.radians(cfg.lidar_half_fov_deg)))
pulse_proxy=lidar_hits*nominal_pulses
coverage=pd.DataFrame([(name,int(mask.sum()),int((mask&(hits>0)).sum())) for name,mask,hits in [("Land LiDAR",~water,lidar_hits),("Required RGB",rgb_required,rgb_hits)]],columns=["Screen","Cells tested","Cells covered"])
coverage["Covered %"]=100*coverage["Cells covered"]/coverage["Cells tested"]; display(coverage)
summary=pd.DataFrame([("Field days",field_days,"days"),("Field time",total_min/60,"h"),("Sorties",len(sorties),"sorties"),("Airborne time",sorties["Airborne min"].sum(),"min"),("Survey distance",acquisition.distance_m.sum()/1000,"km"),("RGB images",photos.images.sum(),"images"),("Mean single-pass emitted density",nominal_pulses,"pulses/m²"),("Maximum GSD proxy",cfg.gsd_max_cm,"cm/pixel"),("Maximum slant proxy",plan["reach"],"m")],columns=["Metric","Value","Unit"])
display(summary.round(3)); summary.to_csv(output_dir/'mission_summary.csv',index=False); coverage.to_csv(output_dir/'coverage_summary.csv',index=False)
pd.DataFrame(dict(east_m=points[:,0],north_m=points[:,1],water=water,rgb_required=rgb_required,lidar_strips=lidar_hits,rgb_strips=rgb_hits,rgb_individual_images=rgb_frames,nominal_emitted_density=pulse_proxy,first_lidar_s=np.where(np.isfinite(first_l),first_l,np.nan),first_rgb_s=np.where(np.isfinite(first_r),first_r,np.nan))).to_csv(output_dir/'coverage_grid.csv',index=False)

## 4. Route, coverage, and schedule plots
Elapsed time is summed on-site working time; overnight breaks are not represented. No driving time is assumed because the field clock begins at the parking lot.

In [ ]:
COLORS = {"IMU allowance":"#7364a4","Survey": "#137b79", "Outbound transit": "#cb9040", "Return transit": "#cb9040",
          "Reposition": "#bdad91", "Turn": "#bdad91", "Climb": "#7364a4", "Descent": "#7364a4",
          "Turnaround and data QA": "#5e7694", "Setup and checks": "#74834b",
          "Survey control": "#9aa971", "Delay allowance": "#a5acad", "Packdown": "#74834b"}

def map_base(ax):
    ax.set_facecolor("#f1f1e5")
    ax.fill(*cfg.lake_xy.T, color="#cbe5ef", zorder=0)
    ax.fill(*cfg.island_xy.T, color="#f1f1e5", zorder=0)
    ax.plot(*cfg.polygon.T, color="#2e4755", lw=1.3)
    ax.plot(*cfg.trail.T, color="#946a42", lw=1.1)
    ax.scatter(*base, c="#1f2633", s=35, zorder=7)
    ax.annotate(site_id, base, xytext=(-5, -15), textcoords="offset points", ha="right",
                fontweight="bold", zorder=7)
    ax.set(aspect="equal", xlabel="Local east (m)", ylabel="Local north (m)")
    for name, point in zip(cfg.corner_names, cfg.corners):
        ax.annotate(name, point, xytext=(4, 4), textcoords="offset points")

def save_figure(fig, name):
    fig.savefig(output_dir / name, dpi=200, facecolor="white")
    if show_plots:
        plt.show()
    plt.close(fig)

fig, ax = plt.subplots(figsize=(8.5, 7), layout="constrained")
map_base(ax)
for number, row in enumerate(sorties.itertuples(index=False, name=None)):
    day_id, sortie_id = row[:2]
    color = plt.cm.tab10(number % 10)
    legs = timeline[(timeline.day == day_id) & (timeline.sortie == sortie_id)]
    for leg in legs.itertuples():
        active = leg.lidar_on
        ax.plot([leg.east_start_m, leg.east_end_m], [leg.north_start_m, leg.north_end_m],
                color=color, alpha=1 if active else 0.24, lw=1.4 if active else 0.7)
    ax.plot([], [], color=color, label=f"D{day_id} · Sortie {sortie_id}")
ax.set_title("Natadesleen Lake survey route")
ax.legend(loc="lower left", fontsize=8)
save_figure(fig, "mission_route.png")

fig, axes = plt.subplots(1, 2, figsize=(12, 6), layout="constrained")
for ax, hits, eligible, title in zip(axes, (lidar_hits, rgb_hits),
                                    (~water, rgb_required),
                                    ("Terrestrial LiDAR swaths", "RGB acquisition strips")):
    map_base(ax)
    plotted = ax.scatter(*points[eligible].T, c=np.minimum(hits[eligible], 6), s=4, marker="s",
                         cmap="YlGnBu", vmin=0, vmax=6, linewidths=0, rasterized=True)
    gaps = eligible & (hits == 0)
    if gaps.any():
        ax.scatter(*points[gaps].T, s=10, c="#e04444", zorder=6, label="Geometric gap")
        ax.legend()
    ax.set_title(title)
fig.colorbar(plotted, ax=axes, label="Acquisition strips per cell (6 = 6 or more)", shrink=0.65)
save_figure(fig, "mission_coverage.png")

fig, (top, bottom) = plt.subplots(2, 1, figsize=(12, 6), layout="constrained",
                                  gridspec_kw={"height_ratios": [1, 1.4]})
for leg in timeline.itertuples():
    t0 = leg.start_s / 60
    width = (leg.end_s - leg.start_s) / 60
    top.broken_barh([(t0, width)], (0, 1), facecolors=COLORS[leg.phase])
for row in sorties.itertuples(index=False, name=None):
    day_id, sortie_id, _, _, takeoff, landing, *_ = row
    top.text((takeoff + landing) / 2, 1.08, f"D{day_id} S{sortie_id}", ha="center", fontsize=9)
top.set(xlim=(0, total_min), ylim=(0, 1.35), yticks=[], xlabel="Elapsed field time (min)",
        title="Full acquisition campaign")
first = timeline[(timeline.day == 1) & (timeline.sortie == 1) &
                 (timeline.phase != "Turnaround and data QA")]
origin = first.start_s.iloc[0]
for leg in first.itertuples():
    times = (np.array([leg.start_s, leg.end_s]) - origin) / 60
    bottom.axvspan(*times, color=COLORS[leg.phase], alpha=0.28)
    bottom.plot(times, [leg.height_start_m, leg.height_end_m], color="#253443", lw=2)
bottom.set(xlim=(0, (first.end_s.iloc[-1] - origin) / 60), ylim=(0, 1.2*cfg.flight_height_above_base_m),
           ylabel="Height above launch (m)", xlabel="Time since takeoff (min)",
           title="Sortie 1 | LiDAR and RGB active in teal")
legend_phases = ["Setup and checks", "Survey control", "Survey", "Outbound transit",
                 "Reposition", "Climb", "Turnaround and data QA", "Delay allowance"]
legend_labels = ["Setup / packdown", "Survey control", "LiDAR + RGB", "Transit",
                 "Turns / reposition", "Climb / descent", "Battery swap / QA", "Delay allowance"]
fig.legend([Line2D([], [], color=COLORS[p], lw=6) for p in legend_phases], legend_labels,
           ncol=4, fontsize=8, loc="lower center", bbox_to_anchor=(0.5,-0.02))
save_figure(fig, "mission_timeline.png")

In [ ]:
fig,ax=plt.subplots(figsize=(9,4),layout="constrained")
t=np.linspace(0,timeline.end_s.iloc[-1],500)
for label,mask,arrival in [("LiDAR land",~water,first_l),("RGB required",rgb_required,first_r)]:
    ax.plot(t/60,100*np.searchsorted(np.sort(arrival[mask]),t,side="right")/mask.sum(),label=label)
ax.set(xlabel="Elapsed field minutes",ylabel="Geometric coverage (%)",ylim=(0,101)); ax.legend(); save_figure(fig,"coverage_progress.png")

5. Presentation playback

The MP4 shows sortie 1 from takeoff to landing, then the complete acquisition campaign including ground activities. Each segment is linearly time-compressed. The visible clock follows the simulation. The aircraft marker turns teal during LiDAR/RGB acquisition and the altitude indicator uses the simulated flat height. The completed survey strips accumulate in teal. Set export_video in the first code cell and rerun for a fresh MP4.

In [ ]:
import shutil
import matplotlib as mpl
from pathlib import Path
import sys
ffmpeg=shutil.which("ffmpeg") or str(Path(sys.prefix)/"Library/bin/ffmpeg.exe")
if export_video:
    if not Path(ffmpeg).exists(): raise RuntimeError("Install FFmpeg or disable export_video")
    mpl.rcParams["animation.ffmpeg_path"]=ffmpeg

In [ ]:
if export_video:
    fps = 12
    detail_times = np.linspace(first.start_s.iloc[0], first.end_s.iloc[-1], 12 * fps)
    campaign_times = np.linspace(0, timeline.end_s.iloc[-1], 22 * fps)
    playback = np.concatenate([detail_times, campaign_times])
    ends = timeline.end_s.to_numpy()
    survey_ends = acquisition.end_s.to_numpy()

    fig, ax = plt.subplots(figsize=(12.8, 7.2), dpi=100)
    fig.subplots_adjust(left=0.08, right=0.66, bottom=0.12, top=0.86)
    map_base(ax)
    for strip in route:
        ax.plot(*strip.T, color="#86999d", lw=0.8, alpha=0.4)
    drawn = [ax.plot(*strip.T, color=COLORS["Survey"], lw=1.4, visible=False)[0] for strip in route]
    active_line, = ax.plot([], [], color=COLORS["Survey"], lw=2.8, zorder=6)
    aircraft, = ax.plot([], [], "o", mec="white", mew=1.5, ms=9, zorder=8)
    fig.text(0.08, 0.93, "Natadesleen Lake | Mission simulation", fontsize=21, weight="bold")
    segment_title = fig.text(0.69, 0.76, "", fontsize=14, weight="bold")
    readout = fig.text(0.69, 0.68, "", fontsize=13, va="top", linespacing=1.6)
    fig.text(0.69, 0.22, "Teal indicates data acquisition\nand completed survey strips.", fontsize=11)

    def update(frame):
        t = playback[frame]
        leg = timeline.iloc[min(np.searchsorted(ends, t, side="right"), len(timeline) - 1)]
        duration = leg.end_s - leg.start_s
        fraction = np.clip((t - leg.start_s) / duration, 0, 1) if duration else 1
        p0 = leg[["east_start_m", "north_start_m", "height_start_m"]].to_numpy(dtype=float)
        p1 = leg[["east_end_m", "north_end_m", "height_end_m"]].to_numpy(dtype=float)
        x, y, z = p0 + fraction * (p1 - p0)
        aircraft.set_data([x], [y])
        aircraft.set_color(COLORS["Survey"] if leg.lidar_on else "#273d57")
        complete = np.searchsorted(survey_ends, t, side="right")
        for i, line in enumerate(drawn):
            line.set_visible(i < complete)
        active_line.set_data([p0[0], x], [p0[1], y]) if leg.lidar_on else active_line.set_data([], [])
        detail = frame < len(detail_times)
        segment_title.set_text("Sortie 1 | Launch to landing" if detail else "Full field campaign")
        clock = t - detail_times[0] if detail else t
        speedup = ((detail_times[-1] - detail_times[0]) / 12 if detail else campaign_times[-1] / 22)
        readout.set_text(f"Elapsed  {clock / 60:6.1f} min   ({speedup:.0f}×)\n"
                         f"Day {leg.day}  /  {f'Sortie {leg.sortie}' if leg.sortie else 'Ground'}\n"
                         f"{leg.phase}\nHeight  {z:.0f} m\n"
                         f"LiDAR + RGB  {'ON' if leg.lidar_on else 'OFF'}\n"
                         f"Survey strips  {complete}/{len(route)}")
    movie = FuncAnimation(fig, update, frames=len(playback), interval=1000 / fps, cache_frame_data=False)
    movie.save(output_dir / "mission_playback.mp4",
               writer=FFMpegWriter(fps=fps, codec="libx264", extra_args=["-pix_fmt", "yuv420p", "-movflags", "+faststart"]))
    plt.close(fig)